# Stage 3 · Evaluation

An "exam with solutions": 15 questions, each with the lecture and page(s) where the answer is. We give the exam to two versions of the system and compare the grades.

- **baseline**: like the course notebook. No cleaning, no deduplication, no merging of animation builds.
- **ours**: with all three improvements.

We evaluate the **retrieval** (did we find the right slide?), not Gemini's wording: retrieval is the part we improved, and it is stable from run to run.

**Metrics**
- **Hit@1**: the right page is the *first* result.
- **Hit@5**: the right page is among the *five* results (the ones Gemini receives).

In [ ]:
import json
from rag_core import build_chunks_from_folder, load_or_compute_embeddings, search, embed_texts, pages_label

## 1 · The exam

12 questions written from the slides, 3 written by me. Most of them avoid the words used in the slides, like a real student would.

In [ ]:
questions = json.loads(open("eval_questions.json", encoding="utf-8").read())

for i, q in enumerate(questions, start=1):
    solution = "; ".join(f"{a['document']} p. {a['pages']}" for a in q["answers"])
    print(f"{i:2}. [{q['author']}] {q['question']}\n    -> {solution}")

## 2 · Two versions of the system

Same code, same embedding model: only the three switches change. Each version needs its own embeddings (computed once, then cached).

The embedding model is chosen in `rag_core.py` (`EMBEDDING_BACKEND`). We use the local one (`nomic-embed-text` with Ollama): no daily limit, so we can compute both versions.

In [ ]:
versions = {
    "baseline": build_chunks_from_folder("data/rag_papers", clean=False, dedupe=False, merge=False),
    "ours":     build_chunks_from_folder("data/rag_papers", clean=True,  dedupe=True,  merge=True),
}
embeddings = {
    "baseline": load_or_compute_embeddings(versions["baseline"], name="course_baseline"),
    "ours":     load_or_compute_embeddings(versions["ours"], name="course"),   # same cache as Stage 2
}

for name, chunks in versions.items():
    print(f"{name}: {len(chunks)} chunks")

## 3 · Take the exam

Each question is embedded **once** and reused for both versions (fewer API calls, and a fairer comparison).

In [ ]:
def is_correct(chunk, q):
    """A result is correct if it comes from a right lecture and covers at least one right page."""
    return any(
        chunk["document"] == a["document"] and set(chunk["pages"]) & set(a["pages"])
        for a in q["answers"]
    )


question_vectors = embed_texts([q["question"] for q in questions], verbose=False, kind="query")

results = {}
for name in versions:
    rows = []
    for q, vector in zip(questions, question_vectors):
        found = search(q["question"], versions[name], embeddings[name], top_k=5, question_vector=vector)
        ranks = [rank for rank, (_, chunk) in enumerate(found, start=1) if is_correct(chunk, q)]
        rows.append({"rank": ranks[0] if ranks else None})   # position of the first right result
    results[name] = rows

## 4 · Grades

In [ ]:
def summary(rows):
    n = len(rows)
    hit1 = sum(r["rank"] == 1 for r in rows) / n
    hit5 = sum(r["rank"] is not None for r in rows) / n
    return hit1, hit5

print(f"{'version':<10} {'Hit@1':>7} {'Hit@5':>7}")
for name, rows in results.items():
    hit1, hit5 = summary(rows)
    print(f"{name:<10} {hit1:>7.0%} {hit5:>7.0%}")

## 5 · Question by question

`rank` = position of the first right result (1 = best, `-` = not found in the top 5).

In [ ]:
print(f"{'#':>2}  {'baseline':>8}  {'ours':>4}  question")
for i, q in enumerate(questions):
    b = results["baseline"][i]["rank"] or "-"
    o = results["ours"][i]["rank"] or "-"
    print(f"{i + 1:>2}  {b:>8}  {o:>4}  {q['question'][:70]}")

### What to look at

- Where do the two versions differ? Open the slides of those questions and try to explain why.
- Which questions fail in **both** versions? Those show the limits of the system (e.g. information that is only in an image).